In [2]:
# Noun-only Didakta pipeline: trains on nouns using didakta_1 (cases) and saves errors/context under ex1
import os
import re
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# --- portable paths: repo root found via .git, so this runs on any machine ---
from pathlib import Path
def _find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    return p
REPO = _find_repo()
EXP_DIR = REPO / "experiments" / "ex1-noun-tagging"
DATA = REPO / "data"
RESULTS = EXP_DIR / "results"
MODELS = EXP_DIR / "models"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
train_candidates = [
    "Odyssey5- Didakta_no_unmatched.csv",
    "Iliad1- Didakta_no_unmatched.csv",
]
train_paths = [os.path.join(DATA, name) for name in train_candidates if os.path.exists(os.path.join(DATA, name))]
if not train_paths:
    raise FileNotFoundError("Could not find Odyssey/Iliad training CSV files.")

# helper to normalize tag text
def normalize_tag_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace(".", "")
    text = re.sub(r"\d+$", "", text)
    return text.strip()

# helper to build feature text from a row
def row_to_text(row, columns):
    parts = []
    for column in columns:
        value = row.get(column, "")
        if pd.isna(value):
            continue
            continue
        text = str(value).strip()
        if text:
            parts.append(f"{column}={text}")
    return " | ".join(parts)

# load tag list to identify case tags, articles, pronouns
tag_list_path = os.path.join(DATA, "didakta-tag-list.csv")
tag_list_df = pd.read_csv(tag_list_path, dtype=str)
case_tags = set(tag_list_df.loc[~tag_list_df["is_category"].astype(str).str.lower().eq("true"), "didakta_tag"].dropna().astype(str))
article_tags = set(tag_list_df.loc[tag_list_df["category"].eq("Article"), "didakta_tag"].dropna().astype(str))
pronoun_tags = set(tag_list_df.loc[tag_list_df["category"].astype(str).str.contains("pronoun", case=False, na=False), "didakta_tag"].dropna().astype(str))

# load training data
train_df = pd.concat([pd.read_csv(path, dtype=str) for path in train_paths], ignore_index=True)
for col in ["didakta_1", "didakta_2", "didakta_3"]:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna("").astype(str).map(normalize_tag_text)
    else:
        train_df[col] = ""

# determine POS column
if "pos" not in train_df.columns and "postag" not in train_df.columns:
    raise ValueError("No POS column found in the training data.")
pos_col = "pos" if "pos" in train_df.columns else "postag"

# select noun rows only, require didakta_1 to be a case tag, and exclude articles/pronouns
noun_mask = train_df[pos_col].fillna("").astype(str).str.lower().str.startswith("n")
article_mask = train_df["didakta_1"].isin(article_tags)
pronoun_mask = train_df["didakta_1"].isin(pronoun_tags)
case_mask = train_df["didakta_1"].isin(case_tags)

noun_df = train_df[noun_mask & case_mask & ~article_mask & ~pronoun_mask].copy()

print(f"Noun rows kept: {len(noun_df)}")
if len(noun_df) == 0:
    raise ValueError("No noun rows with case labels found after filtering.")

# choose feature columns (exclude didakta columns)
exclude_cols = {"didakta", "didakta_1", "didakta_2", "didakta_3", "didakta_4"}
feature_cols = [c for c in noun_df.columns if c not in exclude_cols and c not in ["primary_annotators", "secondary_annotators"]]
if not feature_cols:
    feature_cols = [c for c in ["ref", "greek", "lemma", pos_col, "rel", "syn", "line"] if c in noun_df.columns]

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# build X and y (pandas Series to preserve indices)
X = noun_df.apply(lambda row: row_to_text(row, feature_cols), axis=1).astype(str)
y = noun_df["didakta_1"].astype(str)

# restrict to most frequent labels to avoid extreme sparsity
max_classes = None  # Use all available labels
label_counts = Counter(y)
if max_classes is not None and len(label_counts) > max_classes:
    keep_labels = {label for label, _ in label_counts.most_common(max_classes)}
    keep_mask = y.isin(keep_labels)
    X = X[keep_mask]
    y = y[keep_mask]
    noun_df = noun_df[keep_mask].copy()
    print(f"Restricted noun task to top {max_classes} labels.")
else:
    print(f"Using all {len(label_counts)} available noun labels.")

print(f"Noun-task rows: {len(y)}")
print(f"Noun-task unique labels: {len(set(y))}")

# split while preserving original indices so we can map errors back to noun_df
try:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------- Optional transformer-based features (Sentence-BERT) ----------
transformer_available = False
transformer_model = None
try:
    from sentence_transformers import SentenceTransformer
    # try compact SBERT models (may download if not present)
    for cand in ['sentence-transformers/all-mpnet-base-v2', 'sentence-transformers/paraphrase-mpnet-base-v2']:
        try:
            transformer_model = SentenceTransformer(cand)
            transformer_available = True
            print(f'Loaded transformer embedder: {cand}')
            break
        except Exception:
            transformer_model = None
    if not transformer_available:
        print('SentenceTransformer import succeeded but no pretrained candidate loaded.')
except Exception:
    transformer_available = False

class TransformerEstimator:
    """Simple wrapper that encodes texts with a SentenceTransformer and fits a sklearn classifier."""
    def __init__(self, embedder, clf):
        self.embedder = embedder
        self.clf = clf
    def fit(self, X, y):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = self.embedder.encode(texts, show_progress_bar=False)
        self.clf.fit(embs, y)
        return self
    def predict(self, X):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = self.embedder.encode(texts, show_progress_bar=False)
        return self.clf.predict(embs)

# evaluate multiple models including RandomForest and MultinomialNB; tfidf pipelines operate on text, transformer ones on embeddings
models_to_eval = {
    'LinearSVC_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LinearSVC(max_iter=5000))]),
    'LogisticRegression_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=2000, n_jobs=-1))]),
    'MultinomialNB_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', MultinomialNB())]),
    'RandomForest_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced'))]),
}

if transformer_available and transformer_model is not None:
    try:
        models_to_eval['SBERT_Logistic'] = TransformerEstimator(transformer_model, LogisticRegression(max_iter=2000, n_jobs=-1))
        models_to_eval['SBERT_RandomForest'] = TransformerEstimator(transformer_model, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'))
    except Exception as e:
        print(f'Could not instantiate SBERT estimators: {e}')

eval_results = []
for name, m in models_to_eval.items():
    print(f'\nTraining {name}...')
    # Fit: pipelines expect array-like of texts; transformer wrappers accept Series/list
    try:
        if isinstance(m, SKPipeline):
            m.fit(X_tr.values, y_tr.values)
            y_pred = m.predict(X_val.values)
        else:
            m.fit(X_tr, y_tr)
            y_pred = m.predict(X_val)
    except Exception as e:
        print(f'  Error training {name}: {e}')
        continue
    acc = accuracy_score(y_val.values, y_pred)
    bal = balanced_accuracy_score(y_val.values, y_pred)
    prec = precision_score(y_val.values, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_val.values, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val.values, y_pred, average='macro', zero_division=0)
    eval_results.append({
        'model': name,
        'accuracy': acc,
        'balanced_accuracy': bal,
        'macro_precision': prec,
        'macro_recall': rec,
        'macro_f1': f1,
    })
    print(f'  Accuracy: {acc:.4f}')
    print(f'  Balanced Accuracy: {bal:.4f}')
    print(f'  Macro Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}')

results_df = pd.DataFrame(eval_results).sort_values('balanced_accuracy', ascending=False)
print('\nModel comparison:')
print(results_df.to_string(index=False))

if results_df.empty:
    raise RuntimeError('No models trained successfully.')

best_model_name = results_df.iloc[0]['model']
print(f'\nBest model for nouns: {best_model_name}')

# Use the best trained estimator to compute validation errors (do this BEFORE refitting on full data)
best_trained = models_to_eval[best_model_name]
if isinstance(best_trained, SKPipeline):
    val_preds = best_trained.predict(X_val.values)
else:
    val_preds = best_trained.predict(X_val)

# Build validation dataframe preserving original indices
val_df = pd.DataFrame({
    'feature_text': X_val.astype(str).values,
    'true_didakta': y_val.astype(str).values,
}, index=X_val.index)
val_df['pred_didakta'] = val_preds
val_df['is_error'] = val_df['true_didakta'] != val_df['pred_didakta']

acc_val = accuracy_score(val_df['true_didakta'], val_df['pred_didakta'])
bal_val = balanced_accuracy_score(val_df['true_didakta'], val_df['pred_didakta'])
print(f'\nValidation accuracy (best pipeline): {acc_val:.4f}, balanced_accuracy: {bal_val:.4f}')

# Extract error rows and merge with original noun_df for full context
errors_idx = val_df.index[val_df['is_error']].tolist()
errors_df = noun_df.loc[errors_idx].copy() if errors_idx else pd.DataFrame()
if not errors_df.empty:
    errors_df = errors_df.assign(
        true_didakta = val_df.loc[errors_idx, 'true_didakta'].values,
        pred_didakta = val_df.loc[errors_idx, 'pred_didakta'].values,
        feature_text = val_df.loc[errors_idx, 'feature_text'].values,
    )

# build context: for each error row extract a window of surrounding rows from noun_df
context_rows = []
window = 2
for idx in errors_idx:
    try:
        pos = noun_df.index.get_loc(idx)
    except KeyError:
        continue
    start = max(0, pos - window)
    end = min(len(noun_df), pos + window + 1)
    slice_df = noun_df.iloc[start:end].copy()
    slice_df['_error_center_index'] = idx
    context_rows.append(slice_df)

context_df = pd.concat(context_rows, ignore_index=False) if context_rows else pd.DataFrame()

# save CSVs with ex1 prefix into results/
errors_path = os.path.join(RESULTS, 'ex1_noun_errors.csv')
context_path = os.path.join(RESULTS, 'ex1_noun_error_context.csv')
errors_df.to_csv(errors_path, index=True, encoding='utf-8-sig')
context_df.to_csv(context_path, index=True, encoding='utf-8-sig')

print(f'\nSaved {len(errors_df)} error rows to: {errors_path}')
print(f'Saved context rows ({len(context_df)}) to: {context_path}')

# Now fit final model on all noun rows for production use
final_pipe = None
best_model_obj = models_to_eval[best_model_name]
if isinstance(best_model_obj, SKPipeline):
    final_pipe = SKPipeline([
        ('tfidf', TfidfVectorizer(max_features=8000, ngram_range=(1,2))),
        ('clf', best_model_obj.named_steps['clf']),
    ])
    final_pipe.fit(X.values, y.values)
else:
    final_pipe = best_model_obj
    try:
        final_pipe.fit(X.values, y.values)
    except Exception:
        final_pipe.fit(X, y)

# expose variables in notebook for inspection
noun_errors_df = errors_df
noun_error_context_df = context_df
print('\nDone. Inspect `noun_errors_df` and `noun_error_context_df`.')

Noun rows kept: 1519
Feature columns (7): ['ref', 'greek', 'lemma', 'pos', 'rel', 'syn', 'line']
Using all 42 available noun labels.
Noun-task rows: 1519
Noun-task unique labels: 42

Training LinearSVC_tfidf...


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Accuracy: 0.7796
  Balanced Accuracy: 0.3599
  Macro Precision: 0.4030, Recall: 0.3486, F1: 0.3608

Training LogisticRegression_tfidf...
  Accuracy: 0.7007
  Balanced Accuracy: 0.1996
  Macro Precision: 0.2268, Recall: 0.1996, F1: 0.1913

Training MultinomialNB_tfidf...
  Accuracy: 0.5592
  Balanced Accuracy: 0.0729
  Macro Precision: 0.0768, Recall: 0.0729, F1: 0.0608

Training RandomForest_tfidf...
  Accuracy: 0.7500
  Balanced Accuracy: 0.3347
  Macro Precision: 0.3962, Recall: 0.3347, F1: 0.3411

Model comparison:
                   model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1
         LinearSVC_tfidf  0.779605           0.359880         0.402983      0.348633  0.360769
      RandomForest_tfidf  0.750000           0.334693         0.396186      0.334693  0.341149
LogisticRegression_tfidf  0.700658           0.199632         0.226825      0.199632  0.191349
     MultinomialNB_tfidf  0.559211           0.072917         0.076773      0.072917  0.060813

c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Done. Inspect `noun_errors_df` and `noun_error_context_df`.


In [3]:
# Baseline for noun labels: majority-class and stratified dummy classifiers
from sklearn.dummy import DummyClassifier

baseline_models = {
    'most_frequent': DummyClassifier(strategy='most_frequent'),
    'stratified': DummyClassifier(strategy='stratified', random_state=42),
}

baseline_results = []
for name, clf in baseline_models.items():
    clf.fit(np.zeros((len(X_tr), 1)), y_tr.values)
    baseline_pred = clf.predict(np.zeros((len(X_val), 1)))
    acc = accuracy_score(y_val.values, baseline_pred)
    bal = balanced_accuracy_score(y_val.values, baseline_pred)
    prec = precision_score(y_val.values, baseline_pred, average='macro', zero_division=0)
    rec = recall_score(y_val.values, baseline_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val.values, baseline_pred, average='macro', zero_division=0)
    baseline_results.append({
        'model': name,
        'accuracy': acc,
        'balanced_accuracy': bal,
        'macro_precision': prec,
        'macro_recall': rec,
        'macro_f1': f1,
    })
    print(f'{name}: accuracy={acc:.4f}, balanced_accuracy={bal:.4f}, macro_f1={f1:.4f}')

baseline_results_df = pd.DataFrame(baseline_results).sort_values('balanced_accuracy', ascending=False)
print('\nBaseline comparison:')
print(baseline_results_df.to_string(index=False))

most_frequent: accuracy=0.2993, balanced_accuracy=0.0323, macro_f1=0.0149
stratified: accuracy=0.1513, balanced_accuracy=0.0772, macro_f1=0.0456

Baseline comparison:
        model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1
   stratified  0.151316           0.077189         0.039750      0.070378  0.045650
most_frequent  0.299342           0.032258         0.009656      0.032258  0.014863


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
# Fine-tune Ancient Greek BERT on Didakta noun labels (with HF token from .env)
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# --- portable paths: repo root found via .git, so this runs on any machine ---
from pathlib import Path
def _find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    return p
REPO = _find_repo()
EXP_DIR = REPO / "experiments" / "ex1-noun-tagging"
DATA = REPO / "data"
RESULTS = EXP_DIR / "results"
MODELS = EXP_DIR / "models"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
# ============= Read HF token from .env =============
hf_token = None
env_path = os.path.join(REPO, '.env')
if os.path.exists(env_path):
    try:
        with open(env_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ('HUGGINGFACE_TOKEN', 'HF_TOKEN', 'HF_HUB_TOKEN', 'HF') and v:
                    hf_token = v
                    break
    except Exception as e:
        print(f'Error reading .env: {e}')

if hf_token:
    print(f'Using HF token from .env (prefix: {hf_token[:20]}...)')
else:
    print('No HF token found in .env; proceeding without auth')

# ============= Configuration =============
model_candidates = [
    'pranaydeeps/Ancient-Greek-BERT',
    'nlpaueb/bert-base-greek-uncased-v1',
    'bert-base-multilingual-cased',
]
quick_run = False
sample_frac = 1.0  # Use all data
max_length = 128
per_device_train_batch_size = 8
per_device_eval_batch_size = 16
num_train_epochs = 3 if not quick_run else 1
learning_rate = 2e-5

# ============= Prepare data =============
try:
    X_train_texts = X_tr.astype(str).to_numpy() if hasattr(X_tr, 'to_numpy') else X_tr.astype(str).values
    y_train = y_tr.astype(str).to_numpy() if hasattr(y_tr, 'to_numpy') else y_tr.astype(str).values
    X_val_texts = X_val.astype(str).to_numpy() if hasattr(X_val, 'to_numpy') else X_val.astype(str).values
    y_val = y_val.astype(str).to_numpy() if hasattr(y_val, 'to_numpy') else y_val.astype(str).values
    print('Using train/val split from Cell 1.')
except Exception as e:
    print(f'Error loading split: {e}; creating new split')
    from sklearn.model_selection import train_test_split
    X_arr = X.astype(str).to_numpy() if hasattr(X, 'to_numpy') else X.astype(str).values
    y_arr = y.astype(str).to_numpy() if hasattr(y, 'to_numpy') else y.astype(str).values
    try:
        X_train_texts, X_val_texts, y_train, y_val = train_test_split(X_arr, y_arr, test_size=0.2, random_state=42, stratify=y_arr)
    except ValueError:
        X_train_texts, X_val_texts, y_train, y_val = train_test_split(X_arr, y_arr, test_size=0.2, random_state=42)

# Subsample for quick CPU runs
if quick_run:
    import pandas as _pd
    tr_df = _pd.DataFrame({'text': X_train_texts, 'label': y_train})
    tr_df = tr_df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    X_train_texts = tr_df['text'].values
    y_train = tr_df['label'].values
    print(f'Quick-run: training on {len(y_train)} examples')

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(np.concatenate([y_train, y_val]))
num_labels = len(label_encoder.classes_)
label2id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id2label = {str(idx): label for label, idx in label2id.items()}
print(f'Labels: {num_labels} unique')

# ============= Load model and tokenizer =============
selected_model = None
for cand in model_candidates:
    try:
        print(f'Loading: {cand}...')
        kwargs = {'token': hf_token} if hf_token else {}
        tokenizer = AutoTokenizer.from_pretrained(cand, **kwargs)
        config = AutoConfig.from_pretrained(cand, num_labels=num_labels, label2id=label2id, id2label=id2label, **kwargs)
        model = AutoModelForSequenceClassification.from_pretrained(cand, config=config, **kwargs)
        selected_model = cand
        print(f'✓ Loaded: {cand}')
        break
    except Exception as e:
        print(f'✗ {cand}: {str(e)[:60]}...')

if selected_model is None:
    raise RuntimeError('Could not load any model.')

# ============= Prepare datasets =============
def to_hf_dataset(texts, labels):
    return Dataset.from_dict({
        'text': texts.tolist() if hasattr(texts, 'tolist') else list(texts),
        'label': [label2id[l] for l in labels]
    })

train_ds = to_hf_dataset(X_train_texts, y_train)
val_ds = to_hf_dataset(X_val_texts, y_val)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=max_length)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print(f'Datasets: train={len(train_ds)}, val={len(val_ds)}')

# ============= Metrics =============
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'macro_precision': prec, 'macro_recall': rec, 'macro_f1': f1}

# ============= Training =============
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

out_dir = os.path.join(MODELS, 'ancient_greek_bert_ex1')
training_args = TrainingArguments(
    output_dir=out_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=learning_rate,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()

# Save model
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
print(f'✓ Model saved to: {out_dir}')

# ============= Predictions =============
print('\nGenerating predictions...')
pred_out = trainer.predict(val_ds)
pred_ids = np.argmax(pred_out.predictions, axis=1)
pred_labels = [id2label[str(int(i))] for i in pred_ids]
true_labels = [label_encoder.classes_[i] for i in pred_out.label_ids]

import pandas as pd
bert_val_df = pd.DataFrame({
    'feature_text': X_val_texts,
    'true_didakta': true_labels,
    'pred_didakta': pred_labels,
})
val_out_path = os.path.join(RESULTS, 'ex1_ancientbert_val_preds.csv')
bert_val_df.to_csv(val_out_path, index=False, encoding='utf-8-sig')
print(f'✓ Saved: {val_out_path}')

print('\n=== Classification Report (Validation) ===')
print(classification_report(true_labels, pred_labels, zero_division=0))

Using HF token from .env (prefix: hf_GXQAfJBANgKMGKwwV...)
Error loading split: 'numpy.ndarray' object has no attribute 'values'; creating new split
Labels: 40 unique
Loading: pranaydeeps/Ancient-Greek-BERT...


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Farnoosh\.cache\huggingface\hub\models--pranaydeeps--Ancient-Greek-BERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2566.31it/s]
[transformers] Be

✓ Loaded: pranaydeeps/Ancient-Greek-BERT


Map: 100%|██████████| 304/304 [00:00<00:00, 3452.94 examples/s]


Datasets: train=1213, val=304
Device: cpu
Starting fine-tuning...


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,1.850318,1.642273,0.608553,0.089366,0.135264,0.099472
2,1.184764,1.238570,0.710526,0.157100,0.227707,0.179643
3,1.108798,1.167071,0.717105,0.177581,0.239612,0.194660


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.91s/it]
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✓ Model saved to: c:\Users\Farnoosh\Documents\GitHub\Didakta\ancient_greek_bert_ex1

Generating predictions...


✓ Saved: c:\Users\Farnoosh\Documents\GitHub\Didakta\ex1_ancientbert_val_preds.csv

=== Classification Report (Validation) ===
              precision    recall  f1-score   support

      AccCog       0.00      0.00      0.00         1
    AccDirec       0.00      0.00      0.00         5
      AccObj       0.88      0.99      0.93        92
     AccResp       0.67      0.33      0.44         6
     AccSubj       0.00      0.00      0.00         2
     AccTime       0.00      0.00      0.00         4
     DatAdvn       0.00      0.00      0.00         3
    DatAgent       0.00      0.00      0.00         1
     DatCaus       0.00      0.00      0.00         2
     DatExpr       0.00      0.00      0.00         2
    DatInstr       0.00      0.00      0.00         9
      DatLoc       0.32      0.67      0.43        12
     DatMean       0.00      0.00      0.00         1
     DatVerb       0.58      0.92      0.71        12
     GenCaus       0.00      0.00      0.00         2
     GenC

In [ ]:
# Save the trained BERT model and export the fine-tuning results
import os
import json
import pandas as pd

# --- portable paths: repo root found via .git, so this runs on any machine ---
from pathlib import Path
def _find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    return p
REPO = _find_repo()
EXP_DIR = REPO / "experiments" / "ex1-noun-tagging"
DATA = REPO / "data"
RESULTS = EXP_DIR / "results"
MODELS = EXP_DIR / "models"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
# Save the model/tokenizer again so this can be run independently after training
if 'trainer' in globals() and 'out_dir' in globals():
    trainer.save_model(out_dir)
    if 'tokenizer' in globals():
        tokenizer.save_pretrained(out_dir)
    print(f'Saved model to: {out_dir}')
else:
    print('Trainer or output directory not found; model was not re-saved.')

# Save validation predictions if the BERT dataframe exists.
if 'bert_val_df' in globals() and isinstance(bert_val_df, pd.DataFrame):
    val_out_path = os.path.join(RESULTS, 'ex1_ancientbert_val_preds.csv')
    bert_val_df.to_csv(val_out_path, index=False, encoding='utf-8-sig')
    print(f'Saved validation results to: {val_out_path}')
else:
    print('bert_val_df not found; results CSV was not written.')

# Save model comparison table from Cell 1 if it exists
if 'results_df' in globals() and isinstance(results_df, pd.DataFrame):
    results_out_path = os.path.join(RESULTS, 'ex1_noun_model_comparison.csv')
    results_df.to_csv(results_out_path, index=False, encoding='utf-8-sig')
    print(f'Saved model comparison to: {results_out_path}')

# Save a compact run summary for reuse
summary = {
    'selected_model': globals().get('selected_model', None),
    'best_model_name': globals().get('best_model_name', None),
    'num_labels': int(globals().get('num_labels', 0)) if 'num_labels' in globals() else None,
    'out_dir': globals().get('out_dir', None),
    'validation_rows': int(len(bert_val_df)) if 'bert_val_df' in globals() and isinstance(bert_val_df, pd.DataFrame) else None,
}
summary_path = os.path.join(RESULTS, 'ex1_ancientbert_run_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'Saved run summary to: {summary_path}')

Trainer or output directory not found; model was not re-saved.
Saved validation results to: c:\Users\Farnoosh\Documents\GitHub\Didakta\ex1_ancientbert_val_preds.csv
Saved model comparison to: c:\Users\Farnoosh\Documents\GitHub\Didakta\ex1_noun_model_comparison.csv
Saved run summary to: c:\Users\Farnoosh\Documents\GitHub\Didakta\ex1_ancientbert_run_summary.json


In [10]:
# Ancient Greek BERT evaluation: report the main classification metrics on the BERT validation output
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report
import os
import pandas as pd

# --- portable paths: repo root found via .git, so this runs on any machine ---
from pathlib import Path
def _find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    return p
REPO = _find_repo()
EXP_DIR = REPO / "experiments" / "ex1-noun-tagging"
DATA = REPO / "data"
RESULTS = EXP_DIR / "results"
MODELS = EXP_DIR / "models"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
# Prefer the live BERT validation predictions if the fine-tuning cell has already run.
if 'trainer' in globals() and 'val_ds' in globals():
    bert_eval_source = 'live trainer.predict(val_ds)'
    bert_pred_out = trainer.predict(val_ds)
    bert_pred_ids = bert_pred_out.predictions.argmax(-1)

    if 'label_encoder' in globals() and 'id2label' in globals():
        bert_true = pd.Series([label_encoder.classes_[i] for i in bert_pred_out.label_ids], dtype=str)
        bert_pred = pd.Series([id2label[str(int(i))] for i in bert_pred_ids], dtype=str)
    else:
        bert_true = pd.Series(bert_pred_out.label_ids.astype(str), dtype=str)
        bert_pred = pd.Series(bert_pred_ids.astype(str), dtype=str)
else:
    bert_eval_source = 'saved ex1_ancientbert_val_preds.csv'
    bert_preds_path = os.path.join(RESULTS, 'ex1_ancientbert_val_preds.csv')
    if not os.path.exists(bert_preds_path):
        raise FileNotFoundError(f'Could not find BERT validation predictions at {bert_preds_path}')
    bert_eval_df = pd.read_csv(bert_preds_path, dtype=str)
    bert_true = bert_eval_df['true_didakta'].astype(str)
    bert_pred = bert_eval_df['pred_didakta'].astype(str)

bert_metrics = {
    'accuracy_score': accuracy_score(bert_true, bert_pred),
    'balanced_accuracy_score': balanced_accuracy_score(bert_true, bert_pred),
    'precision_score_macro': precision_score(bert_true, bert_pred, average='macro', zero_division=0),
    'recall_score_macro': recall_score(bert_true, bert_pred, average='macro', zero_division=0),
    'f1_score_macro': f1_score(bert_true, bert_pred, average='macro', zero_division=0),
}

bert_metrics_df = pd.DataFrame([{
    'model': 'Ancient Greek BERT',
    'source': bert_eval_source,
    **bert_metrics,
}])

print('Ancient Greek BERT evaluation metrics:')
print(bert_metrics_df[['model', 'accuracy_score', 'balanced_accuracy_score', 'precision_score_macro', 'recall_score_macro', 'f1_score_macro']].to_string(index=False))
print(f"\nEvaluation source: {bert_eval_source}")
print(f"Validation rows: {len(bert_true)}")
print('\nClassification report:')
print(classification_report(bert_true, bert_pred, zero_division=0))

# Keep the per-row outputs available for inspection.
bert_results_df = pd.DataFrame({
    'true_didakta': bert_true,
    'pred_didakta': bert_pred,
})
bert_results_df['is_error'] = bert_results_df['true_didakta'] != bert_results_df['pred_didakta']
print('\nTop rows:')
print(bert_results_df.head().to_string(index=False))

Ancient Greek BERT evaluation metrics:
             model  accuracy_score  balanced_accuracy_score  precision_score_macro  recall_score_macro  f1_score_macro
Ancient Greek BERT        0.779605                  0.35988               0.402983            0.348633        0.360769

Evaluation source: saved ex1_ancientbert_val_preds.csv
Validation rows: 304

Classification report:
              precision    recall  f1-score   support

      AccAdv       0.00      0.00      0.00         1
      AccCog       0.00      0.00      0.00         1
    AccDirec       0.00      0.00      0.00         2
    AccDoubl       0.00      0.00      0.00         4
      AccObj       0.82      0.98      0.89        91
     AccResp       1.00      0.75      0.86         4
     AccSubj       1.00      1.00      1.00         2
     AccTime       1.00      0.50      0.67         4
     DatAdvn       0.00      0.00      0.00         3
    DatAgent       0.00      0.00      0.00         1
     DatCaus       0.00    

c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
